# Edit 173 K BF4 Bond-Distance Distribution Plot

This notebook reloads the saved 173 K EMI-BF4 bond-distance samples and regenerates the B-F distribution plot. Edit the style, binning, labels, axis limits, or output path directly in the cells below.

Expected input files are produced by:

```bash
conda activate mace_latest_env
python scripts/analysis/run_bf4_173k_bond_distribution.py
```


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "results" / "bf4_173k_distribution").exists():
    ROOT = Path.cwd().parents[1]

RESULTS = ROOT / "results" / "bf4_173k_distribution"
ASSETS = ROOT / "assets"
SAMPLES_CSV = RESULTS / "bf4_173K_bond_distance_samples.csv"
SUMMARY_CSV = RESULTS / "bf4_173K_bond_distance_summary.csv"

samples = pd.read_csv(SAMPLES_CSV)
summary = pd.read_csv(SUMMARY_CSV)

print(f"Loaded {len(samples)} distance samples")
print(SAMPLES_CSV)
summary.round(4)


In [ ]:
# Reference values from Laws et al. and crystallography.
DFT_BF = {
    "B1-F1": 1.364,
    "B1-F2": 1.365,
    "B1-F3": 1.378,
    "B1-F4": 1.703,
}

EXPERIMENT_BF = {
    "B1-F1": 1.376,
    "B1-F2": 1.386,
    "B1-F3": 1.391,
    "B1-F4": 1.399,
}

BONDS = ["B1-F1", "B1-F2", "B1-F3", "B1-F4"]
METHODS = ["MACE-medium", "MACE-POLAR-1"]
COLORS = {"MACE-medium": "#0072B2", "MACE-POLAR-1": "#D55E00"}


In [ ]:
def configure_plot_style():
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times", "Nimbus Roman", "DejaVu Serif"],
        "font.size": 15,
        "font.weight": "bold",
        "legend.fontsize": 11,
        "axes.linewidth": 2.0,
        "lines.linewidth": 2.2,
        "axes.labelweight": "bold",
        "axes.titleweight": "bold",
        "xtick.major.size": 6,
        "xtick.top": True,
        "ytick.right": True,
        "xtick.minor.size": 3,
        "xtick.major.width": 1.6,
        "xtick.minor.width": 1.2,
        "xtick.direction": "in",
        "ytick.major.size": 6,
        "ytick.minor.size": 3,
        "ytick.major.width": 1.6,
        "ytick.minor.width": 1.2,
        "ytick.direction": "in",
        "xtick.major.top": True,
        "xtick.minor.top": True,
        "ytick.major.right": True,
        "ytick.minor.right": True,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


## Plot Controls

Edit these values to tune the figure.

In [ ]:
FIGSIZE = (11.0, 8.2)
DPI = 300
N_BINS = 42
X_MIN = 1.30
X_MAX_DEFAULT = 1.52
X_MAX_B1_F4 = 1.74
SHOW_MODEL_MEANS = True
SHOW_TEXT_BOX = True
OUTPUT_STEM = "bf4_173K_bond_distribution_edited"


In [ ]:
def make_plot(save=True):
    configure_plot_style()
    fig, axes = plt.subplots(2, 2, figsize=FIGSIZE, sharey=True)

    for ax, bond in zip(axes.flat, BONDS):
        x_max = X_MAX_B1_F4 if bond == "B1-F4" else X_MAX_DEFAULT
        bins = np.linspace(X_MIN, x_max, N_BINS)

        for method in METHODS:
            values = samples.loc[
                (samples["method"] == method) & (samples["bond"] == bond),
                "distance_A",
            ]
            ax.hist(
                values,
                bins=bins,
                density=True,
                histtype="step",
                linewidth=2.4,
                color=COLORS[method],
                label=method,
            )
            if SHOW_MODEL_MEANS:
                stats = summary[(summary["method"] == method) & (summary["bond"] == bond)].iloc[0]
                ax.axvline(stats["mean"], color=COLORS[method], linestyle="-", linewidth=1.5, alpha=0.75)

        ax.axvline(DFT_BF[bond], color="black", linestyle="--", linewidth=2.0, label="DFT")
        ax.axvline(EXPERIMENT_BF[bond], color="#009E73", linestyle=":", linewidth=2.4, label="Experiment")
        ax.set_title(bond)
        ax.set_xlim(X_MIN, x_max)
        ax.minorticks_on()

        if SHOW_TEXT_BOX:
            ax.text(
                0.04,
                0.92,
                f"DFT {DFT_BF[bond]:.3f} A\nExp {EXPERIMENT_BF[bond]:.3f} A",
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=10,
                fontweight="bold",
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.75, "pad": 2},
            )

    axes[1, 0].set_xlabel("B-F distance (A)")
    axes[1, 1].set_xlabel("B-F distance (A)")
    axes[0, 0].set_ylabel("Probability density")
    axes[1, 0].set_ylabel("Probability density")

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.00))
    fig.suptitle("173 K BF4 Bond-Length Distributions in EMI-BF4", y=0.985, fontsize=21, fontweight="bold")
    fig.tight_layout(rect=(0, 0.06, 1, 0.96))

    if save:
        ASSETS.mkdir(parents=True, exist_ok=True)
        png = ASSETS / f"{OUTPUT_STEM}.png"
        pdf = ASSETS / f"{OUTPUT_STEM}.pdf"
        fig.savefig(png, dpi=DPI)
        fig.savefig(pdf)
        print(f"Saved {png}")
        print(f"Saved {pdf}")
    return fig, axes

fig, axes = make_plot(save=True)


## Numerical Summary

Use this table to report distribution centers and deviations from DFT/experiment.

In [ ]:
summary.round(4)


In [ ]:
summary.groupby("method")[["mean_abs_error_vs_dft_A", "mean_abs_error_vs_experiment_A"]].mean().round(4)
